In [1]:
import pandas as pd
pd.set_option("display.max_rows", 200)   # show all rows
pd.set_option("display.max_columns", 200) # show all columns

def move_column(df, col_name, new_position):
    assert col_name in df.columns, "Column not found in DataFrame"
    assert 0 <= new_position < len(df.columns), "New position out of bounds"
    cols = df.columns.tolist()
    cols.insert(new_position, cols.pop(cols.index(col_name)))
    return df[cols]

In [4]:
from pathlib import Path
from typing import Optional, Tuple, Union
import numpy as np
import pandas as pd
import joblib
import shap

# ---------- Loader ----------
def load_pipeline(model: str = "lgbm", thresholded: bool = True):
    """
    Load a saved sklearn Pipeline from ./models/{model}/{model}_{thrs|unthrs}.joblib
    """
    suffix = "thrs" if thresholded else "unthrs"
    path = Path("./models") / model / f"{model}_{suffix}.joblib"
    if not path.exists():
        raise FileNotFoundError(f"Pipeline file not found at {path}")
    pipeline = joblib.load(path)
    return pipeline, path


# ---------- Internals ----------
def _get_preprocessor(pipeline) -> Optional[object]:
    return getattr(pipeline, "named_steps", {}).get("preprocessor", None)

def _transform_and_names(preproc, X_new: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Transform X_new with preprocessor (if present) and return (X_transformed, feature_names).
    Falls back to raw values and columns if no preprocessor or no names available.
    """
    if preproc is None:
        return X_new.values, np.array(X_new.columns, dtype=object)

    X_tr = preproc.transform(X_new)
    if hasattr(preproc, "get_feature_names_out"):
        feat_names = preproc.get_feature_names_out()
    else:
        # Fallback: unnamed transformed features
        feat_names = np.array([f"f{i}" for i in range(X_tr.shape[1])], dtype=object)
    return X_tr, np.array(feat_names, dtype=object)

def _is_tree_estimator(est) -> bool:
    # LightGBM, XGBoost, CatBoost, sklearn tree/forest/GBDT
    tree_like_classes = (
        "LGBMClassifier", "LGBMRegressor",
        "XGBClassifier", "XGBRegressor",
        "CatBoostClassifier", "CatBoostRegressor",
        "DecisionTreeClassifier", "DecisionTreeRegressor",
        "RandomForestClassifier", "RandomForestRegressor",
        "ExtraTreesClassifier", "ExtraTreesRegressor",
        "GradientBoostingClassifier", "GradientBoostingRegressor",
        "HistGradientBoostingClassifier", "HistGradientBoostingRegressor",
    )
    name = est.__class__.__name__
    return any(name == t for t in tree_like_classes)

def _aggregate_to_original(feat_names: np.ndarray, shap_row: np.ndarray) -> pd.DataFrame:
    """
    Heuristic to collapse transformed feature SHAP values (e.g., one-hot) back to base columns.
    Works with ColumnTransformer naming like 'num__age' or 'cat__sex_F'.
    Adjust parsing if your naming differs.
    """
    base = []
    for s in map(str, feat_names):
        if "__" in s:
            # take RHS after first "__"
            rhs = s.split("__", 1)[1]
            # for one-hot like "sex_F", take base before first "_" (keeps "sex")
            base.append(rhs.split("_", 1)[0])
        else:
            base.append(s)
    base = np.array(base)

    df = pd.DataFrame(shap_row.reshape(1, -1), columns=feat_names)
    agg = (
        df.T
        .assign(__base__=base)
        .groupby("__base__")
        .sum()
        .T
    )
    return agg


# ---------- Public API ----------
def predict_with_shap_from_store(
    model: str,
    thresholded: bool,
    X_new: Union[pd.DataFrame, pd.Series],
    *,
    positive_class_index: int = 1,
    aggregate_back_to_original: bool = True,
    background: Optional[pd.DataFrame] = None,
    return_merged: bool = True,
):
    """
    Load pipeline -> predict proba for one row -> compute SHAP.

    Returns:
      proba: float
      shap_transformed: DataFrame (1 x n_transformed_features)
      shap_original: DataFrame or None (1 x n_original_features), aggregated by base column
      merged_row: DataFrame or None (original features + '<col>_shap' + 'proba'), only if return_merged=True
    """
    # Ensure exactly one row
    if isinstance(X_new, pd.Series):
        X_new = X_new.to_frame().T
    if not isinstance(X_new, pd.DataFrame) or len(X_new) != 1:
        raise ValueError("X_new must be a single-row pandas DataFrame/Series.")

    pipeline, path = load_pipeline(model=model, thresholded=thresholded)
    preproc = _get_preprocessor(pipeline)
    final_est = pipeline[-1]

    # 1) Predict probability (on the raw X_new; pipeline handles preprocessing)
    if hasattr(pipeline, "predict_proba"):
        proba = float(pipeline.predict_proba(X_new)[:, positive_class_index])
    elif hasattr(pipeline, "decision_function"):
        from scipy.special import expit
        proba = float(expit(pipeline.decision_function(X_new)))
    else:
        raise ValueError("Pipeline has neither predict_proba nor decision_function.")

    # 2) Transform to what the model actually sees + names
    X_tr, feat_names = _transform_and_names(preproc, X_new)

    # 3) Compute SHAP
    try:
        if _is_tree_estimator(final_est):
            explainer = shap.TreeExplainer(final_est)
            sv = explainer.shap_values(X_tr)
            # Binary tree models may return list [neg, pos]
            if isinstance(sv, list):
                shap_row = np.array(sv[positive_class_index]).reshape(1, -1)
            else:
                shap_row = np.array(sv).reshape(1, -1)
        else:
            # Model-agnostic fallback (needs background)
            if background is None:
                raise ValueError(
                    "This pipeline's final estimator is not tree-based. "
                    "Provide a small 'background' DataFrame (~50-200 rows) for SHAP."
                )
            if isinstance(background, pd.Series):
                background = background.to_frame().T

            # Define prediction function over original features
            f = lambda X: pipeline.predict_proba(pd.DataFrame(X, columns=background.columns))[:, positive_class_index]
            explainer = shap.PermutationExplainer(f, background)
            shap_row = explainer(X_new)[0].values.reshape(1, -1)
            # In this fallback, feat_names refer to original columns
            feat_names = np.array(X_new.columns, dtype=object)
    except Exception as e:
        raise RuntimeError(f"Failed to compute SHAP: {e}")

    shap_transformed = pd.DataFrame(shap_row, columns=feat_names, index=X_new.index)

    shap_original = None
    if aggregate_back_to_original and shap_transformed.shape[1] > 0:
        shap_original = _aggregate_to_original(feat_names, shap_row)

    merged_row = None
    if return_merged:
        merged_row = X_new.copy()
        # Prefer aggregated if available; otherwise use transformed names
        if shap_original is not None:
            for c in shap_original.columns:
                merged_row[f"{c}_shap"] = shap_original.iloc[0][c]
        else:
            for c in shap_transformed.columns:
                merged_row[f"{c}_shap"] = shap_transformed.iloc[0][c]
        merged_row["proba"] = proba

    return proba, shap_transformed, shap_original, merged_row


In [19]:
from pathlib import Path
from typing import Optional, Tuple, List
import pandas as pd
import numpy as np

# --- choose the correct value column from the flag ---
def _pick_value_column(df: pd.DataFrame, thresholded: bool) -> str:
    """
    Decide which column to use as the value source based on the 'thresholded' flag.
    Handles small naming variations robustly.
    """
    # candidate names seen in your data
    candidates_thrs  = ["GM_values_thresholded", "GMvalues_thresholded", "GM_value_thresholded"]
    candidates_unth  = ["GMvalues_unthresholded", "GM_values_unthresholded", "GM_value_unthresholded"]

    candidates = candidates_thrs if thresholded else candidates_unth
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Could not find a value column for thresholded={thresholded}. "
                   f"Looked for: {candidates}")

def _expected_raw_columns_from_pipeline(pipeline) -> Optional[List[str]]:
    """
    Try several places where sklearn stores the original *raw* feature names.
    """
    # sklearn >=1.0 often sets feature_names_in_ on the final estimator and/or preprocessor
    if hasattr(pipeline, "feature_names_in_"):
        return list(pipeline.feature_names_in_)
    preproc = getattr(pipeline, "named_steps", {}).get("preprocessor", None)
    if preproc is not None and hasattr(preproc, "feature_names_in_"):
        return list(preproc.feature_names_in_)
    # As a fallback, return None (we'll just use whatever columns we have)
    return None

def preprocess_seg_df_for_pipeline(
    seg_df: pd.DataFrame,
    pipeline,
    *,
    thresholded: bool,
    key: str = "name",           # or "index" if your pipeline was trained on numeric IDs
    aggfunc: str = "mean",       # how to collapse duplicates per region
) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """
    Convert a per-region long df -> single row with columns matching the pipeline's expected raw inputs.

    seg_df columns expected:
      - 'name' (region string) and/or 'index' (int ID)
      - one of: 'GM_values_thresholded' OR 'GMvalues_unthresholded' (chosen by flag)

    Returns:
      X_new        : (1 x n_raw_features) DataFrame aligned to pipeline expected columns if known
      missing_cols : expected-by-pipeline but absent in seg_df
      extra_cols   : present in seg_df but NOT expected by pipeline (they'll be dropped)
    """
    if key not in seg_df.columns:
        raise KeyError(f"seg_df does not contain the key column '{key}'. Available: {list(seg_df.columns)}")

    val_col = _pick_value_column(seg_df, thresholded)

    # Clean region identifiers
    df = seg_df[[key, val_col]].copy()
    # Ensure 'index' is str if we use it as column header and training used str; keep numeric otherwise
    if key == "index":
        # Many pipelines train with stringified IDs; if yours used plain ints, you can comment the next line.
        df[key] = df[key].astype(str)

    # Aggregate if duplicates exist
    if aggfunc not in {"mean", "median", "first"}:
        raise ValueError("aggfunc must be one of {'mean','median','first'}")
    if aggfunc == "mean":
        s = df.groupby(key, dropna=False)[val_col].mean()
    elif aggfunc == "median":
        s = df.groupby(key, dropna=False)[val_col].median()
    else:
        s = df.groupby(key, dropna=False)[val_col].first()

    # Wide, single row
    X_wide = s.T.to_frame().T  # 1 x K
    X_wide.index = [0]

    # Align to pipeline expected raw columns (if we can discover them)
    expected_cols = _expected_raw_columns_from_pipeline(pipeline)

    if expected_cols is None:
        # We don't know the training-time raw feature list; just return what we built.
        # Your pipeline's preprocessor should be robust to missing/unexpected columns only
        # if it was fit with a ColumnTransformer that selects by name.
        return X_wide, [], []

    # Build a 1 x n_expected frame filled with NaN; fill from X_wide where names match
    X_aligned = pd.DataFrame(index=[0], columns=expected_cols, dtype=float)
    X_aligned.loc[0, list(set(expected_cols).intersection(X_wide.columns))] = \
        X_wide.loc[0, list(set(expected_cols).intersection(X_wide.columns))]

    missing_cols = [c for c in expected_cols if c not in X_wide.columns]
    extra_cols   = [c for c in X_wide.columns if c not in expected_cols]

    # Leave NaNs for missing; the pipeline's imputer should handle them
    return X_aligned, missing_cols, extra_cols

# --- convenience wrapper for a CSV path + model flags (uses your existing loader) ---
from pathlib import Path
import joblib

def make_X_from_segmentation_csv(
    csv_path: str | Path,
    pipeline,
    *,
    thresholded: bool,
    key: str = "name",
    sep : str = ";",
) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """
    Read the segmentation CSV for ONE subject and return X_new aligned to the pipeline.
    """
    seg_df = pd.read_csv(csv_path, sep=sep)
    return preprocess_seg_df_for_pipeline(seg_df, pipeline, thresholded=thresholded, key=key)


In [20]:
df = pd.read_csv("data/sPR04383_NS300459-0011-00001-000176-01_T1w_NM-val-gen_regions_summary.csv", sep=";")

In [21]:
df

,name,index,GMvalues_unthresholded,GMvalues_thresholded
0,3rd Ventricle,4,1.070000,0.180000
1,4th Ventricle,11,1.924000,0.000000
2,Right Accumbens Area,23,0.292000,0.292000
3,Left Accumbens Area,30,0.348000,0.348000
4,Brain Stem,35,20.362999,0.493000
5,Right Caudate,36,2.060000,1.990000
6,Left Caudate,37,2.138000,2.079000
7,Right Cerebellum Exterior,38,54.857997,38.341998
8,Left Cerebellum Exterior,39,56.599997,38.723998
9,Right Cerebral White Matter,44,183.634991,8.760000


In [22]:
# 1) Load your pipeline (from our previous helper)
pipe, pth = load_pipeline(model="lgbm", thresholded=True)

# 2) Build the single-row input from your CSV
X_new, missing, extra = make_X_from_segmentation_csv(
    csv_path="data/sPR04383_NS300459-0011-00001-000176-01_T1w_NM-val-gen_regions_summary.csv",
    pipeline=pipe,
    thresholded=True,   # or False
    key="name"          # use "index" if your pipeline was trained on IDs
)

print("Missing expected columns (filled with NaN -> imputed):", missing[:10], "…", len(missing), "total")
print("Extra columns in CSV (ignored):", extra[:10], "…", len(extra), "total")

# 3) Predict and (optionally) get SHAP using the earlier function
proba, shap_tr, shap_orig, merged = predict_with_shap_from_store(
    model="lgbm",
    thresholded=True,
    X_new=X_new,
    positive_class_index=1,
    aggregate_back_to_original=True,
    return_merged=True
)

print("Probability:", proba)
display(merged)


Missing expected columns (filled with NaN -> imputed): [] … 0 total
Extra columns in CSV (ignored): ['3rd Ventricle', '4th Ventricle', 'CSF', 'Left Inf Lat Vent', 'Left Lateral Ventricle', 'Optic Chiasm', 'Right Inf Lat Vent', 'Right Lateral Ventricle'] … 8 total


/tmp/ipykernel_6187/211946930.py:116: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  proba = float(pipeline.predict_proba(X_new)[:, positive_class_index])


Probability: 0.604466297838914


/home/jovyan/venv/lib/python3.11/site-packages/shap/explainers/_tree.py:583: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/tmp/ipykernel_6187/211946930.py:167: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_row[f"{c}_shap"] = shap_original.iloc[0][c]
/tmp/ipykernel_6187/211946930.py:167: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_row[f"{c}_shap"] = shap_original.iloc[0][c]
/tmp/ipykernel_6187/211946930.py:167: PerformanceWarning: D

,Right Accumbens Area,Left Accumbens Area,Brain Stem,Right Caudate,Left Caudate,Right Cerebellum Exterior,Left Cerebellum Exterior,Right Cerebral White Matter,Left Cerebral White Matter,Right Pallidum,Left Pallidum,Right Putamen,Left Putamen,Right vessel,Left vessel,Cerebellar Vermal Lobules I-V,Cerebellar Vermal Lobules VI-VII,Cerebellar Vermal Lobules VIII-X,Left Basal Forebrain,Right Basal Forebrain,Right ACgG anterior cingulate gyrus,Left ACgG anterior cingulate gyrus,Right AIns anterior insula,Left AIns anterior insula,Right AOrG anterior orbital gyrus,Left AOrG anterior orbital gyrus,Right AnG angular gyrus,Left AnG angular gyrus,Right Calc calcarine cortex,Left Calc calcarine cortex,Right CO central operculum,Left CO central operculum,Right Cun cuneus,Left Cun cuneus,Right Ent entorhinal area,Left Ent entorhinal area,Right FO frontal operculum,Left FO frontal operculum,Right FRP frontal pole,Left FRP frontal pole,Right FuG fusiform gyrus,Left FuG fusiform gyrus,Right GRe gyrus rectus,Left GRe gyrus rectus,Right IOG inferior occipital gyrus,Left IOG inferior occipital gyrus,Right ITG inferior temporal gyrus,Left ITG inferior temporal gyrus,Right LiG lingual gyrus,Left LiG lingual gyrus,Right LOrG lateral orbital gyrus,Left LOrG lateral orbital gyrus,Right MCgG middle cingulate gyrus,Left MCgG middle cingulate gyrus,Right MFC medial frontal cortex,Left MFC medial frontal cortex,Right MFG middle frontal gyrus,Left MFG middle frontal gyrus,Right MOG middle occipital gyrus,Left MOG middle occipital gyrus,Right MOrG medial orbital gyrus,Left MOrG medial orbital gyrus,Right MPoG postcentral gyrus medial segment,Left MPoG postcentral gyrus medial segment,Right MPrG precentral gyrus medial segment,Left MPrG precentral gyrus medial segment,Right MSFG superior frontal gyrus medial segment,Left MSFG superior frontal gyrus medial segment,Right MTG middle temporal gyrus,Left MTG middle temporal gyrus,Right OCP occipital pole,Left OCP occipital pole,Right OFuG occipital fusiform gyrus,Left OFuG occipital fusiform gyrus,Right OpIFG opercular part of the inferior frontal gyrus,Left OpIFG opercular part of the inferior frontal gyrus,Right OrIFG orbital part of the inferior frontal gyrus,Left OrIFG orbital part of the inferior frontal gyrus,Right PCgG posterior cingulate gyrus,Left PCgG posterior cingulate gyrus,Right PCu precuneus,Left PCu precuneus,Right PHG parahippocampal gyrus,Left PHG parahippocampal gyrus,Right PIns posterior insula,Left PIns posterior insula,Right PO parietal operculum,Left PO parietal operculum,Right PoG postcentral gyrus,Left PoG postcentral gyrus,Right POrG posterior orbital gyrus,Left POrG posterior orbital gyrus,Right PP planum polare,Left PP planum polare,Right PrG precentral gyrus,Left PrG precentral gyrus,Right PT planum temporale,Left PT planum temporale,Right SCA subcallosal area,Left SCA subcallosal area,...,Left TTG transverse temporal gyrus_shap,Left TrIFG triangular part of the inferior frontal gyrus_shap,Left agd_astr_shap,Left agd_centromedian_shap,Left agd_laterobasal_shap,Left agd_superficial_shap,Left dentate nucleus_shap,Left hi_SPAM_CA1-3_shap,Left hi_SPAM_CA4-DG_shap,Left hi_SPAM_subiculum_shap,Left red nucleus_shap,Left substantia nigra_shap,Left th_anterior_shap,Left th_geniculate_shap,Left th_intralaminar_shap,Left th_mediodorsal_shap,Left th_posteriorAnterior_shap,Left th_pulvinar_shap,Left th_ventroanterior_shap,Left th_ventrolateral_shap,Left th_ventroposterior_shap,Left vessel_shap,Right ACgG anterior cingulate gyrus_shap,Right AIns anterior insula_shap,Right AOrG anterior orbital gyrus_shap,Right Accumbens Area_shap,Right AnG angular gyrus_shap,Right Basal Forebrain_shap,Right CO central operculum_shap,Right Calc calcarine cortex_shap,Right Caudate_shap,Right Cerebellum Exterior_shap,Right Cerebral White Matter_shap,Right Cun cuneus_shap,Right Ent entorhinal area_shap,Right FO frontal operculum_shap,Right FRP frontal pole_shap,Right FuG fusiform gyrus_shap,Right GRe gyrus rectus_shap,